# Dynamic Unicycle CLF Derivations
 Uses the SymPy differential geometry library to compute the CLF condition to be used in the CLF-based controller (which was ultimately discarded).

 > Note that this file is vestigial and simply shows an example of using the differential geometry library for more than just CBFs, this is not used in producing any of the results in the attached paper.

In [2]:
import sympy as sp
import sympy.diffgeom as dg

In [3]:
state_mfld = dg.Manifold("M", 5)
state_mfld_patch = dg.Patch("P", state_mfld)

x, y, theta, v, omega = sp.symbols(r"x,y,\theta,v,\omega", real=True)

state_mfld_coords = dg.CoordSystem(
    "StateSpace", state_mfld_patch, (x, y, theta, v, omega)
)

(x_sc, y_sc, theta_sc, v_sc, omega_sc) = state_mfld_coords.base_scalars()
(x_vec, y_vec, theta_vec, v_vec, omega_vec) = state_mfld_coords.base_vectors()

In [4]:
# define the vector fields of the system

f = (
    v_sc * sp.cos(theta_sc) * x_vec
    + v_sc * sp.sin(theta_sc) * y_vec
    + omega_sc * theta_vec
)

f1 = v_vec
f2 = omega_vec

display(f, f1, f2)

sin(\theta)*v*e_y + cos(\theta)*v*e_x + \omega*e_\theta

e_v

e_\omega

In [ ]:
# define the variables associated with tthe trajectory

t = sp.symbols("t", real=True)
alpha, beta, mu, lamb = sp.symbols(r"\alpha,\beta,\mu,\lambda", real=True)

x_tr = sp.Function("x_tr")(t)
y_tr = sp.Function("y_tr")(t)
theta_tr = sp.Function(r"\theta_tr")(t)
v_tr = sp.Function("v_tr")(t)
omega_tr = sp.Function(r"\omega")(t)

0.5*\alpha*((-x_tr(t) + x)**2 + (-y_tr(t) + y)**2) + 0.5*\beta*(-\theta_tr(t) + \theta)**2

In [31]:
# define the constraint such that the lyapunov function is considered to be
# a -ve barrier function so can use higher order construction

# define the constraint to enforce decreasing error
v_fn = (
    0.5 * alpha * ((x_sc - x_tr) ** 2 + (y_sc - y_tr) ** 2)
    + 0.5 * beta * (theta_sc - theta_tr) ** 2
    # + 0.5 * mu * (v - v_tr)**2
    # + 0.5 * lamb * (omega - omega_tr)
)
k1, k2 = sp.symbols("k1,k2", nonneg=True)

phi_0 = v_fn
phi_1 = sp.diff(phi_0, t) + k1 * phi_0
phi_2 = sp.diff(phi_1, t) + k2 * phi_1

Lf_Lf_v = dg.LieDerivative(f, dg.LieDerivative(f, v_fn)).nsimplify().simplify()

Lf1_Lf_v = dg.LieDerivative(f1, dg.LieDerivative(f, v_fn)).nsimplify().simplify()
Lf2_Lf_v = dg.LieDerivative(f2, dg.LieDerivative(f, v_fn)).nsimplify().simplify()

d2v_dt2 = sp.diff(v_fn, t, 2).nsimplify().simplify()

o_v_fn = (dg.LieDerivative(f, k1 * phi_0) + sp.diff(k1 * phi_0, t)).nsimplify().simplify()
final_term = (k2 * phi_1).nsimplify().simplify()

u_f, u_t = sp.symbols("F,T", real=True)
g_constraint = (
    Lf_Lf_v + Lf1_Lf_v * u_f + Lf2_Lf_v * u_t + d2v_dt2 + o_v_fn + final_term
).nsimplify().simplify() <= 0

display(v_fn)
display("Components of constraint:", Lf_Lf_v, Lf1_Lf_v, Lf2_Lf_v, d2v_dt2, o_v_fn, final_term)
display("Final constraint:", g_constraint)

0.5*\alpha*((-x_tr(t) + x)**2 + (-y_tr(t) + y)**2) + 0.5*\beta*(-\theta_tr(t) + \theta)**2

'Components of constraint:'

\alpha*sin(\theta)**2*v**2 + \alpha*cos(\theta)**2*v**2 + (\alpha*(x_tr(t) - x)*sin(\theta)*v - \alpha*(y_tr(t) - y)*cos(\theta)*v + \beta*\omega)*\omega

\alpha*((-x_tr(t) + x)*cos(\theta) + (-y_tr(t) + y)*sin(\theta))

\beta*(-\theta_tr(t) + \theta)

\alpha*((x_tr(t) - x)*Derivative(x_tr(t), (t, 2)) + (y_tr(t) - y)*Derivative(y_tr(t), (t, 2)) + Derivative(x_tr(t), t)**2 + Derivative(y_tr(t), t)**2) + \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), (t, 2)) + \beta*Derivative(\theta_tr(t), t)**2

k1*(\alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) - \alpha*(x_tr(t) - x)*cos(\theta)*v - \alpha*(y_tr(t) - y)*sin(\theta)*v - \beta*(\theta_tr(t) - \theta)*\omega + \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t))

k2*(2*\alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) + 2*\beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t) + k1*(\alpha*((x_tr(t) - x)**2 + (y_tr(t) - y)**2) + \beta*(\theta_tr(t) - \theta)**2))/2

'Final constraint:'

-F*\alpha*((x_tr(t) - x)*cos(\theta) + (y_tr(t) - y)*sin(\theta)) - T*\beta*(\theta_tr(t) - \theta) + \alpha*((x_tr(t) - x)*Derivative(x_tr(t), (t, 2)) + (y_tr(t) - y)*Derivative(y_tr(t), (t, 2)) + Derivative(x_tr(t), t)**2 + Derivative(y_tr(t), t)**2) + \alpha*sin(\theta)**2*v**2 + \alpha*cos(\theta)**2*v**2 + \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), (t, 2)) + \beta*Derivative(\theta_tr(t), t)**2 - k1*(-\alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) + \alpha*(x_tr(t) - x)*cos(\theta)*v + \alpha*(y_tr(t) - y)*sin(\theta)*v + \beta*(\theta_tr(t) - \theta)*\omega - \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t)) + k2*(2*\alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) + 2*\beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t) + k1*(\alpha*((x_tr(t) - x)**2 + (y_tr(t) - y)**2) + \beta*(\theta_tr(t) - \theta)**2))/2 + (\alpha*(x_tr(t) - x)*sin(\theta)*v - \alpha*(y_tr(t) - y)*cos(\theta)*v

In [10]:
Lf_v = dg.LieDerivative(f, v_fn)
Lf1_v = dg.LieDerivative(f1, v_fn)
Lf2_v = dg.LieDerivative(f2, v_fn)

display(Lf_v)
display(Lf1_v)
display(Lf2_v)


0.5*\alpha*(-2*x_tr(t) + 2*x)*cos(\theta)*v + 1.0*\alpha*(-y_tr(t) + y)*sin(\theta)*v + 1.0*\beta*(-\theta_tr(t) + \theta)*\omega

0

0

In [ ]:
# define the inputs


# define the optimization problem

p, delta = sp.symbols(r"p,delta", real=True)
prob_f = (0.5 * (u_f**2 + u_t**2) + p * delta**2).simplify()
prob_g = (Lf_Lf_v + Lf1_Lf_v * u_f + Lf2_Lf_v * u_t + dv_dt <= delta).simplify()
prob_vars = [u_f, u_t, delta]

display(prob_f)
display(prob_g)

0.5*F**2 + 0.5*T**2 + delta**2*p

delta >= -F*\alpha*((x_tr(t) - x)*cos(\theta) + (y_tr(t) - y)*sin(\theta)) - T*\beta*(\theta_tr(t) - \theta) + \alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) + \alpha*sin(\theta)**2*v**2 + \alpha*cos(\theta)**2*v**2 + \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t) + (\alpha*(x_tr(t) - x)*sin(\theta)*v - \alpha*(y_tr(t) - y)*cos(\theta)*v + \beta*\omega)*\omega